### 3 dimesions tables flights,passengers(customers) and airports
### 1 fact table bookings and all numeric columns

In [0]:
# #key col list

# key_cols = "['flight_id']"
# key_cols_list = eval(key_cols)

# #cdc_col(chnage data capture) like modified date when data is updated or changed
# cdc_col = 'modifiedDate'
# source_schema = 'silver'
# source_object = 'silver_flights'

# target_schema = 'gold'

# target_object = 'DIMflights'

# backdated_refresh = ''

# surrogate_key = 'DIMflights_id_key'


In [0]:
# #key col list

# key_cols = "['airport_id']"
# key_cols_list = eval(key_cols)

# #cdc_col(chnage data capture) like modified date when data is updated or changed
# cdc_col = 'modifiedDate'
# source_schema = 'silver'
# source_object = 'silver_airports'

# target_schema = 'gold'

# target_object = 'DIMairports'

# backdated_refresh = ''

# surrogate_key = 'DIMairports_id_key'


In [0]:
# #key col list

# key_cols = "['passenger_id']"
# key_cols_list = eval(key_cols)

# #cdc_col(chnage data capture) like modified date when data is updated or changed
# cdc_col = 'modifiedDate'
# source_schema = 'silver'
# source_object = 'silver_passengers'

# target_schema = 'gold'

# target_object = 'DIMpassengers'

# backdated_refresh = ''

# surrogate_key = 'DIMpassengers_id_key'


### ## last_load_date   for incremental ingestion

In [0]:
if len(backdated_refresh) ==  0:
    #if table exists in destination
    if spark.catalog.tableExists(f"workspace.{target_schema}.{target_object}"):
        last_load = spark.sql(f"select max({cdc_col}) from {target_schema}.{target_object}").collect()[0][0]
    
    else:
        last_load = '1900-01-01 00:00:00'
else:
    last_load = backdated_refresh


last_load
    

In [0]:
key_cols_string_init = [f"'' AS {i}" for i in key_cols_list]
key_cols_string_init

In [0]:
df_src = spark.sql(f"select * from {source_schema}.{source_object} where {cdc_col} >= '{last_load}'")

In [0]:
df_src.display()

### old vs new records


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.types import LongType

In [0]:
if spark.catalog.tableExists(f"workspace.{target_schema}.{target_object}"):
    key_cols_string_incremental = ','.join(key_cols_list) 
    df_target = spark.sql(f"select {key_cols_string_incremental},{surrogate_key},created_date,updated_date from workspace.{target_schema}.{target_object}")
else: 
    key_cols_string_init = [f"'' AS {i}" for i in key_cols_list]
    key_cols_string_init =',' .join(key_cols_string_init)

    df_target = spark.sql(f"""select {key_cols_string_init},CAST('0' AS BIGINT) AS {surrogate_key},
                          CAST('1900-01-01 00:00:00' AS timestamp) AS created_date,CAST('1900-01-01 00:00:00' AS timestamp) AS updated_date  WHERE 1=0""")
     

In [0]:
df_target.display()

#### join condition

In [0]:
join_condition = ' AND '.join ([f"src.{i} = trg.{i}" for i in key_cols_list])
join_condition

In [0]:
df_src.createOrReplaceTempView('src')
df_target.createOrReplaceTempView('trg')

df_join = spark.sql(f"""
          SELECT src.*,trg.{surrogate_key},trg.created_date,trg.updated_date FROM 
          src LEFT JOIN trg ON  {join_condition}
          """)

In [0]:
df_join.display()

In [0]:
#old records will have values which is non null
df_old = df_join.filter(col(f'{surrogate_key}').isNotNull())

# new records will have null in incremental 
df_new = df_join.filter(col(f'{surrogate_key}').isNull())

In [0]:
df_old.display()
df_new.display()

### preparing old df 

How values will be filled in the new columns like surrogate key ,created at and updated
here created_at and surogate key wont be worked on as they are alreday set in df_new as created and key will will set only once. while we update date for updated_at as it is updated when values are updated or changed
 


In [0]:
df_old_enr = df_old.withColumn('updated_date',current_timestamp())

In [0]:
display(df_old_enr.select("updated_date"))

### preparing df_new

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, lit, current_timestamp

if spark.catalog.tableExists(f"workspace.{target_schema}.{target_object}"):

    max_surrogate_key = spark.sql(f"""
        SELECT COALESCE(MAX({surrogate_key}), 0) AS max_surrogate_key
        FROM workspace.{target_schema}.{target_object}
    """).collect()[0][0]

else:
    max_surrogate_key = 0

window = Window.orderBy(lit(1))

df_new_enr = (
    df_new
    .withColumn(
        surrogate_key,
        row_number().over(window) + lit(max_surrogate_key)
    )
    .withColumn("created_date", current_timestamp())
    .withColumn("updated_date", current_timestamp())
)

In [0]:
df_new_enr.display()

##### union for updating in old and new df to merge


In [0]:
df_union = df_old_enr.unionByName(df_new_enr)
df_union.display()

## UPSERT

In [0]:
from delta.tables import DeltaTable

here we first check if table exists or not

if table exists we Open an existing Delta table stored at the specified path.
and alias it as target as all data is going in it from df_union which is source and based on surroagte key we perform merge operation.

if surrogate key matches then record is already there in target so we update it
if surrogate does not match then we insert the record


however if table table does not exist as it might be inital load then
we write/ create a delta table at the specified path 

- Uses the Spark DataFrame writer.
- Stores data using the Delta Lake format.
- Delta format provides:
- ACID transactions
- Schema enforcement
- Time travel
- MERGE/UPSERT support

In [0]:
spark.sql(f"DESCRIBE DETAIL workspace.{target_schema}.{target_object}").display()

In [0]:
if spark.catalog.tableExists(f"workspace.{target_schema}.{target_object}"):
    
    dlt_obj = DeltaTable.forName(spark,f"workspace.{target_schema}.{target_object}")
    dlt_obj.alias('trg').merge(df_union.alias('src'),f"trg.{surrogate_key} = src.{surrogate_key}")\
            .whenMatchedUpdateAll(condition = f"src.{cdc_col} >= trg.{cdc_col}")\
            .whenNotMatchedInsertAll()\
            .execute()
else:
    df_union.write.format("delta")\
            .mode('append')\
            .saveAsTable(f"workspace.{target_schema}.{target_object}")

In [0]:
%sql
select * from workspace.gold.dimpassengers
order by passenger_id